# A3 · GridSearchCV from scratch

**Type**: Implement from scratch · **Level**: ●●○ independent · **Time**: about 35 minutes · **core path ★**

## Objective

Implement a grid search with cross-validation and a final refit, and verify that it gives the same scores as `GridSearchCV`
on the same folds, including the `step__parameter` syntax.

## Dataset

`ds.loans(with_missing=False)`: the four numeric columns and the target `defaulted`. Model: pipeline with `StandardScaler` and
`KNeighborsClassifier`. Grid: `n_neighbors` in `[5, 15, 30, 60]` and `weights` in `["uniform", "distance"]` (8 combinations).

## Educational purpose

See `GridSearchCV` for what it is: a double loop (combinations × folds) with a `set_params` and a mean. You will also understand
why `refit` exists and why it costs so many evaluations.

## Task

1. Write `my_grid_search(estimator, grid, X, y, folds)`: build all the combinations with `itertools.product` on the keys of the grid; for each one clone the estimator, apply `set_params(**combination)`, compute the mean AUC over the `folds` (train/val pairs) and collect the results in a DataFrame `results` (one column per parameter and `mean_score`, sorted from the best).
2. The function returns `(results, best_params, best_model)`, where `best_model` is the estimator with the best parameters **retrained on all the data** (refit).
3. Call `my_grid_search` and compare with `GridSearchCV(estimator, grid, cv=folds, scoring="roc_auc")`: same scores, same best parameters.
4. Save `n_trainings`: the number of times the model is trained in total (combinations × folds, plus the refit).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import itertools

from sklearn.base import clone
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = ds.loans(with_missing=False)
X = df[["age", "income", "job_tenure", "amount"]].to_numpy()
y = df["defaulted"].to_numpy()
estimator = make_pipeline(StandardScaler(), KNeighborsClassifier())
grid = {"kneighborsclassifier__n_neighbors": [5, 15, 30, 60], "kneighborsclassifier__weights": ["uniform", "distance"]}
folds = list(StratifiedKFold(5, shuffle=True, random_state=0).split(X, y))

## Your work

In [ ]:
def my_grid_search(estimator, grid, X, y, folds):
    keys = list(grid)
    rows = []
    for values in itertools.product(*grid.values()):
        params = dict(zip(keys, values))
        ...
    results = pd.DataFrame(rows).sort_values("mean_score", ascending=False).reset_index(drop=True)
    best = ...
    best_model = ...
    return results, best, best_model

In [ ]:
results, best, best_model = my_grid_search(estimator, grid, X, y, folds)
sk = GridSearchCV(estimator, grid, cv=folds, scoring="roc_auc").fit(X, y)
n_trainings = ...
results.round(4)

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='Inside the loop: `m = clone(estimator).set_params(**params)`; for every `(tr, va)`: `m2 = clone(m).fit(X[tr], y[tr])` and the AUC on `va`. `rows.append({**params, "mean_score": np.mean(scores)})`. For the best parameters: `{k: results.loc[0, k] for k in keys}` (mind the types: `int(...)` for `n_neighbors`).'):
    check(len(results) == 8 and "mean_score" in results.columns, "8 combinations with the mean score", "one row per combination")
    check(results["mean_score"].is_monotonic_decreasing, "results sorted from the best", "sort by mean_score descending")
    check(best == sk.best_params_, "the same best parameters as GridSearchCV", "best = dictionary of the parameters of the first row")
    check(close(np.sort(results["mean_score"].to_numpy()), np.sort(sk.cv_results_["mean_test_score"])),
          "the same mean scores as GridSearchCV", "mean of the AUC over the folds, for every combination")
    check(close(best_model.predict_proba(X), sk.best_estimator_.predict_proba(X)), "the refit matches best_estimator_",
          "the final model must be retrained on ALL the data with the best parameters")
    check(n_trainings == 8 * 5 + 1, "41 trainings: 8 combinations x 5 folds + the refit", "combinations x folds, plus one")
done()

---
The commented solution is in `solutions/A_from_scratch/A3_grid_search_from_scratch.ipynb`. Open it after you pass the check, or after you have really tried.